# 🛡️ Machine Learning Lab — Guided Notebook from A to Z

**Goal of the lab**: build an AI that watches the behaviour of a Linux workstation (CPU, RAM, disk, network…) and answers **`allow`** or **`block`** for every event, in real time. When you feel ready, you start **your 5-minute exam**: your AI, on its own, makes decisions on live data.

> ⏱️ **Suggested duration**: 1 preparation session + as many training runs as you like + 1 five-minute exam.
> 🔁 **You can train as much as you want.** You only start the exam **when you feel ready**.

### The path at a glance

```text
 ┌──────────┐   ┌──────────┐   ┌──────────┐   ┌─────────────┐   ┌───────────┐
 │ 1 Setup  │ → │ 2 Data   │ → │ 3 Models │ → │ 4 Train     │ → │ 5 EXAM    │
 │ + paths  │   │ audit    │   │ + thresh.│   │ (dry-run,   │   │  5 min,   │
 │          │   │ split    │   │ + export │   │  training,  │   │  real time│
 └──────────┘   └──────────┘   └──────────┘   │  start over)│   │  ONCE     │
                                              └──────┬──────┘   └───────────┘
                                                     └── ↺ as many times as you like
```

### How to read this notebook

| Symbol | Meaning |
|---|---|
| 📍 | **Paths / files** to use (relative to the project root `tp4/`) |
| ▶️ | **Command / cell** to run |
| ✍️ | **Your turn to write** code (cell with `TODO`) |
| ✅ | **Checkpoint**: a cell tells you whether you can move on |
| 🎓 | **"If you come from AI" tip**: explains the **cyber/security** side |
| 🛡️ | **"If you come from CYBER" tip**: explains the **AI/machine learning** side |
| ⚠️ | Classic pitfall |
| 🏷️ | Domain of the cell: **`AI domain`**, **`CYBER domain`** or **`Mixed`** (used for the permission to use an AI, see right below) |

## 📦 Python dependencies (install them first)

This notebook is **self-contained**: you only need **this file**. The toolbox (feature contract, models, agent, exam) is **inside the notebook**.

| Package | Version | Role |
|---|---|---|
| `torch` | ≥ 2.5 | neural networks (LSTM, GRU, MLP) |
| `numpy` | ≥ 1.26 and < 2.3 | numerical computing |
| `pandas` | ≥ 2.2 and < 3 | data tables |
| `scikit-learn` | ≥ 1.5 and < 2 | baselines, train/validation split, metrics |
| `matplotlib` | ≥ 3.9 and < 4 | plots |
| `psutil` | ≥ 6.1 | measures the CPU / RAM of your agent ("resources" criterion) |

**Requirements**: Python **3.10, 3.11 or 3.12**, JupyterLab (or VS Code), about **3 GB** of free space (PyTorch is heavy), HTTPS access to `https://ml-cyber.magro.pro`. No GPU needed.

▶️ Run the cell below **only once** (a few minutes), then **restart the kernel** (menu *Kernel → Restart Kernel*) before continuing.

> 💡 Prefer a terminal? `python -m pip install "torch>=2.5" "numpy>=1.26,<2.3" "pandas>=2.2,<3" "scikit-learn>=1.5,<2" "matplotlib>=3.9,<4" "psutil>=6.1"` — in **the same environment** as the one Jupyter uses.
> 💡 **Windows**: if you see `WinError 1114 … c10.dll`, it means `torch` was imported after pandas or scikit-learn: restart the kernel and run the start-up cell **first** (it imports `torch` first).

In [1]:
# 📦 Installing the dependencies (only once; PyTorch takes a while to download)
%pip install -q "torch>=2.5" "numpy>=1.26,<2.3" "pandas>=2.2,<3" "scikit-learn>=1.5,<2" "matplotlib>=3.9,<4" "psutil>=6.1"
# ➡️ Then: menu Kernel → Restart Kernel, and continue with the start-up cell.

Note: you may need to restart the kernel to use updated packages.


## 🤖 Permission to use a generative AI (read this before you start)

This lab mixes two professions. Nobody is an expert in both, so:

> ### ✅ You are allowed to use an AI assistant (ChatGPT, Claude, Copilot, Gemini…) **for the parts that are NOT in your domain.**
>
> | Your profile | You may get help from an AI for… | You do **it yourself** (no generative AI) |
> |---|---|---|
> | **You come from CYBER** | the `🏷️ AI domain` cells: pandas, PyTorch, training loop, metrics, sklearn | the `🏷️ CYBER domain` cells: reading behaviours, choosing signals, risks, cost of errors |
> | **You come from AI** | the `🏷️ CYBER domain` cells: understanding ransomware / exfiltration / false positives, choosing attack signals | the `🏷️ AI domain` cells: split, sequences, model, threshold |
> | `🏷️ Mixed` cells | pair work: each of you explains their half to the other | |
>
> In **your own** domain, you can of course read the docs, your course notes and ask your teammate.

### The 5 rules of the game (non-negotiable)

1. **You understand every line you keep.** At the oral, you will be asked to explain any cell.
2. **You record your AI usage** in the log (`log_ai(...)` cell further down): *what you asked, what you kept, how you verified it*.
3. **You verify what it gives you.** Assistants are often wrong here (data leakage, badly built window, misread metric).
4. **You never give it**: your **API key** (`tp_live_…`), the **sealed test set**, or the group's credentials.
5. **During the 5-minute exam, nobody touches anything**: your agent runs on its own. No retraining, no threshold change.

### Ready-to-use prompts (copy, paste, adapt)

| You are… | Suggested prompt |
|---|---|
| CYBER, stuck on pandas/PyTorch | *"I am a cybersecurity student. Explain step by step, with a short example, how to [build sliding windows per group in pandas / write a PyTorch training loop with early stopping]. Don't give me everything at once: one step, then I'll show you my code."* |
| CYBER, doesn't understand a metric | *"Explain precision, recall and F1 with a SOC/antivirus analogy: one false positive, one false negative."* |
| AI, doesn't understand an attack | *"I am in AI. Describe the system behaviour of a ransomware, a data exfiltration and a network scan: which CPU / disk / network / action-rate metrics change, and which legitimate activities look like them (backup, compilation, update)?"* |
| AI, wants signals | *"Which behavioural signals distinguish a legitimate cloud backup from an exfiltration?"* |
| Everyone | *"Review this code: is there data leakage between train and validation?"* |

In [30]:
# 📓 AI usage log (utility cell: run it once)
# Each call appends a line to ai_usage_log.md (hand it in with the lab).
from datetime import datetime
from pathlib import Path

AI_LOG_FILE = Path("ai_usage_log.md")

def log_ai(domaine, demande, gardé, vérification):
    '''domaine: "AI" or "CYBER" (the domain that is NOT yours).'''
    ligne = (f"- {datetime.now():%Y-%m-%d %H:%M} | domain={domaine} | asked: {demande} | "
             f"kept: {gardé} | verified by: {vérification}\n")
    with AI_LOG_FILE.open("a", encoding="utf-8") as fichier:
        fichier.write(ligne)
    print("Logged in", AI_LOG_FILE.resolve())

# Example (delete / replace):
# log_ai("AI", "GroupShuffleSplit syntax", "3 lines", "assert disjoint sessions")

## 🎓🛡️ Two profiles, one shared vocabulary

Keep this little dictionary handy. It translates each world into the other.

| **AI** term | **CYBER** equivalent |
|---|---|
| *feature* (input variable) | observable indicator / signal (behavioural IoC) |
| *label*, ground truth | an analyst's verdict: malicious / legitimate |
| positive class (`1`) | threat to block |
| false positive (FP) | false alarm: a legitimate program blocked (**cost −8**) |
| false negative (FN) | undetected attack (**cost −12**) |
| *recall* | detection rate: share of attacks actually blocked |
| precision | alert reliability: share of alerts that were real attacks |
| decision threshold | sensitivity level of the alert |
| data *leakage* | the AI saw the answer before the exam (unintentional cheating) |
| *overfitting* | the AI memorised the examples, not the logic |
| time window | recent history of a process (event correlation) |
| *baseline* | simple reference rule (like a SIEM rule) |
| *training / serving skew* | the rule tested in the lab is not the one deployed in production |

## 🧭 How you will be assessed (read carefully)

The final assessment is **different** from a classic exam. There is no questionnaire: **your AI sits the test, live**.

### The principle

1. **You train as much as you like, with no key and no limit.** This is the **open training sandbox**: already-labelled events and, for **each** answer, the immediate verdict — **TP / TN / FP / FN** — together with the software family. Nothing is graded, nobody ranks you.
2. **You train your model** (offline), as many times as you like.
3. **When you think your AI is ready, you start the exam yourself**, at the moment you choose, with your group's key:

   ```text
   POST /api/v1/agent/evaluation/start        (X-API-Key header)
   ```
4. After a **countdown of ~10 seconds**, the platform generates a **real-time** stream of events for **5 minutes** on **a simulated workstation that belongs only to your group**.
5. **For each event**, your agent receives the metrics and must answer **exactly once** `allow` or `block` before the deadline (~15 s).
6. At the end of the 5 minutes, the exam ends **automatically**. Your score is frozen and you get your **TP / TN / FP / FN summary**.

### What happens during the 5 minutes

| Point | Detail |
|---|---|
| **Data** | A simulated workstation running ~30 programs **you have never seen** in training, but which belong to the **same behaviour families**. Your AI must **generalise**: learn behaviours, not names. |
| **Volume** | Several events per second: around **600 to 1,200 decisions** in 5 minutes depending on the server configuration. |
| **Feedback** | **None** during the test (unlike the open training sandbox). The TP/TN summary comes **afterwards**. |
| **What do you do?** | **Nothing.** Your agent runs on its own (the cell stays "running" for ~5 min, that is normal). You must not change the model, threshold, code or rules during the test. |
| **No waiting queue** | Each group has **its own simulated workstation**: several groups can take the exam **at the same time**, you have nobody to wait for. |
| **Same test for the whole class** | Groups in the same class receive **exactly the same scenario**: scores are comparable. |
| **How many attempts?** | **One** by default (set by the instructor, **per class**). Think before you start! |
| **Exam closed?** | The instructor can close the class exam (`exam_closed`): keep training while you wait for it to open. |
| **No answer** | An event left unanswered before the deadline **earns no points** and the threat is not blocked. A crashing agent therefore loses points. |

### Reading your decisions: TP / TN / FP / FN

Every decision falls into one of these four boxes (the "confusion matrix"):

| | The software is **malicious** | The software is **legitimate** |
|---|---|---|
| You **block** | **TP** — true positive (API code `VP`) ✅ threat blocked | **FP** — false positive ❌ legitimate blocked by mistake |
| You **allow** | **FN** — false negative ❌ threat let through | **TN** — true negative (API code `VN`) ✅ legitimate allowed |

The **good decisions** are **TP + TN**. You see them live in the open training sandbox, and after the exam via `GET /api/v1/agent/report`.

### How points are computed

| Situation | "Decision" points | "Speed" points (separate) |
|---|---:|---:|
| Threat **blocked** ✅ | **+10** | up to +5 |
| Legitimate activity **allowed** ✅ | **+6** | up to +5 |
| Legitimate activity **blocked** ❌ (false positive) | **−8** | 0 |
| Threat **allowed** ❌ (false negative) | **−12** | 0 |

Speed (measured from the moment **your agent receives** the event, only if the decision is correct): ≤150 ms → 5 pts · ≤500 ms → 4 · ≤1 s → 3 · ≤3 s → 2 · ≤7 s → 1.

> 🛡️ **If you come from CYBER**: this is exactly the dilemma of an EDR. Block everything = zero missed attacks but users can no longer work (−8 each time). Allow everything = everything gets through (−12). The AI must find the balance, and the **threshold** is your sensitivity knob.
>
> 🎓 **If you come from AI**: this scoring is an **asymmetric cost matrix**. The default threshold of 0.5 is not optimal here: an FN costs more than an FP, so we lean slightly towards recall — without blowing up FPs.

### What is graded in the end (indicative grid, the instructor's scale prevails)

| Axis | Weight | Evidence |
|---|---:|---|
| Live decisions (F1, recall, precision, points) | 35 % | 5-minute exam |
| Robustness / false positives | 15 % | exam surprises |
| Speed | 10 % | P50/P95 latency |
| Agent resources (CPU, RAM) | 10 % | agent telemetry |
| Method (leak-free split, baselines, threshold) | 15 % | this notebook |
| Software quality | 5 % | robust agent, no secret in clear text |
| Report / defence | 10 % | conclusion + oral |

## 🎮 Prologue — Prepare your team

Your goal is to improve a detector until the final evaluation. To begin, fill in `GROUP`, `MEMBERS` and `CLASS_CODE` in the next cell, then run the cells in order.

**Starting path:** configuration → connection → game engine → group registration → mission 1.

The server is already set to **`https://ml-cyber.magro.pro`**. The notebook creates the required folders and files automatically.

> 🔐 The registration key is displayed once. Copy it immediately and clear the output before submitting the notebook.

<details><summary>See where files are created</summary>

### Everything is in this notebook

You have **only this file** (`.ipynb`). Put it in an **empty folder** (no spaces or accents in the path, for example `C:\TP\ml` or `~/tp/ml`) and start Jupyter **from that folder**:

```text
jupyter lab
```

The notebook creates by itself what it needs, next to it:

| What is created | Where | What it is for |
|---|---|---|
| `donnees/open-train-42.jsonl` | next to the notebook | dataset downloaded in step 2 (can be regenerated at will) |
| `models/group_XX/` | next to the notebook | your checkpoints: `lstm.pt`, `gru.pt`, `mlp.pt`, **`champion.pt`** |
| `ai_usage_log.md` | next to the notebook | your AI usage log (to hand in) |

</details>

<details><summary>Learn more about the supplied engine</summary>

A collapsed cell, further down, contains everything that is **not** the core of the lab:

| Function | Role |
|---|---|
| `encode`, `padded_sequence`, `SEQUENCE_LENGTH` | **the feature contract**: telemetry → 8 numbers → window of 12 |
| `build_model("lstm" / "gru" / "mlp")` | ready-to-use architectures |
| `save_checkpoint`, `load_checkpoint` | write / reload a model in the format the agent expects |
| `quick_benchmark(...)` | compares the 3 models at once and saves `champion.pt` |
| `run_open_training(...)` | **open training**: your model decides, you see TP/TN/FP/FN right away |
| `run_live(...)`, `start_exam(...)`, `exam_report()` | dress rehearsal, **graded exam**, summary |
| `join_class(...)`, `ask_api_key()` | register your group, enter your key without writing it anywhere |

You may read it later, but you do not need it to get started.

### 🔑 Your API key (never in the notebook)

The official server is **`https://ml-cyber.magro.pro`**. At the start of the lab, your instructor gives you either a **group key** (`tp_live_…`) or a **class code**. With a class code, create your group first by following the commands shown just after the toolbox. The response contains `apiKey`: copy it immediately because it is shown only once.

The key is used for the private training streams, the harmless mini-evaluation and the final exam. Never save it in the submitted notebook. Enter it through the masked field or set the `TP_API_KEY` environment variable.

> 🛡️ **CYBER**: an API key is a secret, like a password. Not in Git, not in a screenshot, not in an AI prompt. If it leaks, it gets revoked and a new one is generated.
> 🎓 **AI**: treat the key like a project `.env`: never hard-coded in the notebook you hand in.

</details>

In [34]:
# 🏷️ Mixed — Start-up cell (run it: it changes nothing important)
import torch   # ⚠️ Windows: import torch BEFORE pandas / sklearn / matplotlib (otherwise a DLL error "WinError 1114")
import os, sys, json, time, hashlib
from pathlib import Path

ROOT = Path.cwd()                                # the folder you started Jupyter from: everything is created here
GROUP = "Cyber_Professional"                               # ✍️ unique group name
MEMBERS = "Govinda BHATTARAI, Ankit BHATT, Muhammad Ali, Ziyad Briand, Dan fotsing"             # ✍️ group members
CLASS_CODE = "A009D9"                        # ✍️ code supplied by the instructor
SEED = 42                                        # ✍️ the official seed given by the instructor
TRAIN_PATH = ROOT / "donnees" / "open-train-42.jsonl"   # downloaded in step 2 (or the instructor's file)
OUT_DIR = ROOT / "models" / GROUP                # your checkpoints
TRAIN_PATH.parent.mkdir(parents=True, exist_ok=True)
if GROUP != "group_XX":
    OUT_DIR.mkdir(parents=True, exist_ok=True)

API_URL = os.getenv("TP_URL", "https://ml-cyber.magro.pro").rstrip("/")
API_KEY = os.getenv("TP_API_KEY") or "tp_live_d8LM82FIf_OX4lDsyqckCefT_SZjXlGH"                # group key; NEVER print or write it in the submitted notebook

def check(nom, condition, aide=""):
    # Prints ✅ / ❌ without crashing the notebook.
    print(("✅ " if condition else "❌ ") + nom + ("" if condition else f"  → {aide}"))
    return bool(condition)

print("Working folder:", ROOT)
check(f"Python ≥ 3.10 (you have {sys.version.split()[0]})", sys.version_info >= (3, 10), "install Python 3.10, 3.11 or 3.12")
print(f"torch {torch.__version__}")
for module in ("numpy", "pandas", "sklearn", "matplotlib"):
    try:
        __import__(module)
        check(f"module {module}", True)
    except ImportError:
        check(f"module {module}", False, "run the \"📦 Dependencies installation\" cell, then restart the kernel")
print("✅ Dataset present" if TRAIN_PATH.exists() else "ℹ️ Dataset missing: it will be downloaded in step 2 (this is normal)")
check("Group name set", GROUP != "group_XX", "replace group_XX in this cell, then re-run it")
print("ℹ️ API key already defined in the environment" if API_KEY else "ℹ️ No key: create/join your group in the next section, then use the masked input")

Working folder: C:\Users\LENOVO\Security_Usescases
✅ Python ≥ 3.10 (you have 3.13.7)
torch 2.15.0+cpu
✅ module numpy
✅ module pandas
✅ module sklearn
✅ module matplotlib
✅ Dataset present
✅ Group name set
ℹ️ API key already defined in the environment


In [35]:
# 🏷️ Mixed — Little tool: talking to the API (read-only here)
import urllib.request, urllib.error

def api(path, payload=None, timeout=5):
    '''GET if payload is None, otherwise JSON POST. Returns (http_code, dict).'''
    data = json.dumps(payload).encode() if payload is not None else None
    req = urllib.request.Request(API_URL + path, data=data,
        headers={"X-API-Key": API_KEY or "", "Content-Type": "application/json"},
        method="POST" if payload is not None else "GET")
    try:
        with urllib.request.urlopen(req, timeout=timeout) as r:
            return r.status, json.load(r)
    except urllib.error.HTTPError as e:
        try:
            return e.code, json.loads(e.read().decode())
        except Exception:
            return e.code, {}
    except OSError as e:
        return 0, {"error": str(e)}

code, health = api("/api/health")
check(f"Platform reachable ({API_URL})", code == 200, f"HTTP {code} {health} — check the URL / the network")
if API_KEY:
    code, snap = api("/api/v1/agent/telemetry")
    check("API key accepted", code == 200, f"HTTP {code} — 401 = invalid key")

✅ Platform reachable (https://ml-cyber.magro.pro)
✅ API key accepted


## ⚙️ Game engine — run it without opening it

The large cell below contains the supplied technical machinery. Simply click **▶**. It does not start training or an exam by itself.

You will use only four commands, explained when needed: `join_class`, `quick_benchmark`, `run_open_training` and `start_exam`.

**Checkpoint:** the message `toolbox ready` must appear.

In [36]:
# 🧰 TOOLBOX / BOÎTE À OUTILS — provided, do not edit (you may read it) / fournie, ne pas modifier (tu peux la lire)
# Contract features · models · checkpoint · quick benchmark · open training · exam agent.
import copy, getpass, hashlib, json, math, os, random, sys, time, urllib.error, urllib.request
from collections import defaultdict, deque
from pathlib import Path
import numpy as np
import torch
from torch import nn
try:
    import psutil
except ImportError:          # optional: only used to report the agent's own CPU / RAM
    psutil = None

LANG = "en"
_TXT = {
    "fr": {
        "TP": "VP ✅ menace bloquée", "TN": "VN ✅ légitime autorisé", "FP": "FP ❌ légitime bloqué", "FN": "FN ❌ menace passée",
        "need_key": "Pas de clé API : exécute d'abord la cellule « 🔑 Ta clé API ».",
        "http": "HTTP {code} : {detail}", "net": "Réseau injoignable : {error}",
        "open_head": "[OUVERT] joueur={player} | aucune clé | rien n'est noté",
        "open_line": "{sw} · {ev} → {dec} (P={p:.2f}) | {outcome} — famille : {cat}",
        "summary": "[BILAN] {n} décisions | VP={vp} VN={vn} FP={fp} FN={fn} | précision={prec:.2f} rappel={rec:.2f} F1={f1:.2f}",
        "stopped": "⏹️ Arrêté. Tu peux relancer la cellule.",
        "state": "[EXAMEN] état={state} | durée={minutes} min | tentatives={used}/{allowed}",
        "joined": "[EXAMEN] ton examen est déjà en cours : l'agent le rejoint (aucune tentative consommée).",
        "unavailable": "Examen impossible maintenant ({state}). Temps restant côté plateforme : {remaining} s.",
        "confirm": "Lancer l'examen NOTÉ de {minutes} min ? Il ne se rejoue pas. Tape {word} pour confirmer : ",
        "word": "EVALUER", "cancel": "Annulé : rien n'a été lancé.",
        "started": "[EXAMEN] lancé : départ dans {delay} s, durée {minutes} min. Ne touche plus au modèle ni au seuil.",
        "progress": "[EXAMEN] {n} décisions envoyées | {blocked} blocages | latence moyenne {lat} ms | reste ~{left} s",
        "done": "[EXAMEN] terminé. décisions={n} points={points} F1={f1}% FP={fp} FN={fn} latence moyenne={lat} ms",
        "report_na": "[EXAMEN] bilan indisponible pour l'instant (HTTP {code}).",
        "cm": "[EXAMEN] matrice de confusion :",
        "cm_tp": "VP (menaces bloquées)", "cm_tn": "VN (légitimes autorisés)", "cm_fp": "FP (légitimes bloqués)", "cm_fn": "FN (menaces laissées)",
        "pts": "[EXAMEN] points={points} F1={f1}%",
        "ckpt_features": "les features du checkpoint ne respectent pas le contrat du TP",
        "ckpt_missing": "checkpoint introuvable : {path}",
        "bench_row": "{name:<5} F1 validation={f1:.3f} précision={prec:.3f} rappel={rec:.3f} seuil={thr:.2f} ({params} paramètres)",
        "bench_champion": "🏆 Champion : {name} → {path}",
        "live_head": "[LIVE] {mode} pendant {seconds} s (session du serveur, nécessite ta clé)",
        "joined_class": "✅ Groupe inscrit dans « {cls} ». ⚠️ NOTE TA CLÉ MAINTENANT (elle ne sera plus affichée) :",
        "key_help": "Colle la clé API de ton groupe (tp_live_…) — Entrée pour continuer sans clé : ",
        "key_ok": "✅ Clé enregistrée pour cette session (elle n'est écrite nulle part).",
        "key_none": "ℹ️ Pas de clé : l'entraînement ouvert fonctionne, l'examen non.",
    },
    "en": {
        "TP": "TP ✅ threat blocked", "TN": "TN ✅ legitimate allowed", "FP": "FP ❌ legitimate blocked", "FN": "FN ❌ threat missed",
        "need_key": "No API key: run the \"🔑 Your API key\" cell first.",
        "http": "HTTP {code}: {detail}", "net": "Network unreachable: {error}",
        "open_head": "[OPEN] player={player} | no key | nothing is graded",
        "open_line": "{sw} · {ev} → {dec} (P={p:.2f}) | {outcome} — family: {cat}",
        "summary": "[SUMMARY] {n} decisions | TP={vp} TN={vn} FP={fp} FN={fn} | precision={prec:.2f} recall={rec:.2f} F1={f1:.2f}",
        "stopped": "⏹️ Stopped. You can re-run the cell.",
        "state": "[EXAM] state={state} | duration={minutes} min | attempts={used}/{allowed}",
        "joined": "[EXAM] your exam is already running: the agent joins it (no attempt used).",
        "unavailable": "Exam not possible right now ({state}). Time left on the platform: {remaining} s.",
        "confirm": "Start the GRADED {minutes}-minute exam? It cannot be replayed. Type {word} to confirm: ",
        "word": "EXAM", "cancel": "Cancelled: nothing was started.",
        "started": "[EXAM] started: begins in {delay} s, lasts {minutes} min. Do not touch the model or the threshold any more.",
        "progress": "[EXAM] {n} decisions sent | {blocked} blocks | average latency {lat} ms | ~{left} s left",
        "done": "[EXAM] finished. decisions={n} points={points} F1={f1}% FP={fp} FN={fn} average latency={lat} ms",
        "report_na": "[EXAM] report not available yet (HTTP {code}).",
        "cm": "[EXAM] confusion matrix:",
        "cm_tp": "TP (threats blocked)", "cm_tn": "TN (legitimate allowed)", "cm_fp": "FP (legitimate blocked)", "cm_fn": "FN (threats missed)",
        "pts": "[EXAM] points={points} F1={f1}%",
        "ckpt_features": "the checkpoint features do not follow the lab contract",
        "ckpt_missing": "checkpoint not found: {path}",
        "bench_row": "{name:<5} validation F1={f1:.3f} precision={prec:.3f} recall={rec:.3f} threshold={thr:.2f} ({params} parameters)",
        "bench_champion": "🏆 Champion: {name} → {path}",
        "live_head": "[LIVE] {mode} for {seconds} s (server session, needs your key)",
        "joined_class": "✅ Group registered in \"{cls}\". ⚠️ WRITE YOUR KEY DOWN NOW (it will not be shown again):",
        "key_help": "Paste your group's API key (tp_live_…) — press Enter to continue without a key: ",
        "key_ok": "✅ Key stored for this session only (it is not written anywhere).",
        "key_none": "ℹ️ No key: open training works, the exam does not.",
    },
}[LANG]


def _t(key, **values):
    return _TXT[key].format(**values)


# ---------------------------------------------------------------------------------------------
# 1) THE CONTRACT: telemetry -> 8 numbers (training and the live agent MUST use the same one)
# ---------------------------------------------------------------------------------------------
FEATURE_NAMES = ("cpu", "ram_mb", "disk_read_mbps", "disk_write_mbps",
                 "network_rx_mbps", "network_tx_mbps", "actions_per_second", "busy")
SEQUENCE_LENGTH = 12


def _bounded(value, scale):
    try:
        return max(0.0, min(2.0, float(value) / scale))
    except (TypeError, ValueError):
        return 0.0


def encode(metrics):
    """dict of metrics -> 8 numbers (normalised, between 0 and 2). Never uses the software name or the truth."""
    return [_bounded(metrics.get("cpu"), 100), _bounded(metrics.get("ramMb"), 512),
            _bounded(metrics.get("diskReadMbps"), 50), _bounded(metrics.get("diskWriteMbps"), 50),
            _bounded(metrics.get("networkRxMbps"), 50), _bounded(metrics.get("networkTxMbps"), 50),
            _bounded(metrics.get("actionsPerSecond"), 250), 1.0 if metrics.get("status") == "busy" else 0.0]


def padded_sequence(history, current):
    """history (previous encoded steps) + current observation -> window of 12 steps (zeros on the left)."""
    values = list(history)
    vector = encode(current)
    if not values or values[-1] != vector:
        values.append(vector)
    padding = [[0.0] * len(FEATURE_NAMES) for _ in range(max(0, SEQUENCE_LENGTH - len(values)))]
    return (padding + values)[-SEQUENCE_LENGTH:]


# ---------------------------------------------------------------------------------------------
# 2) MODELS: lstm / gru / mlp
# ---------------------------------------------------------------------------------------------
class BehavioralLSTM(nn.Module):
    def __init__(self, feature_count=8, hidden_size=40, layers=2, dropout=0.20):
        super().__init__()
        self.input_norm = nn.LayerNorm(feature_count)
        self.lstm = nn.LSTM(feature_count, hidden_size, layers, batch_first=True, dropout=dropout if layers > 1 else 0)
        self.head = nn.Sequential(nn.LayerNorm(hidden_size), nn.Linear(hidden_size, 24), nn.GELU(),
                                  nn.Dropout(dropout), nn.Linear(24, 1))

    def forward(self, sequence):
        output, _ = self.lstm(self.input_norm(sequence))
        return self.head(output[:, -1, :]).squeeze(-1)


class BehavioralGRU(nn.Module):
    def __init__(self, feature_count=8, hidden_size=40, layers=2, dropout=0.20):
        super().__init__()
        self.input_norm = nn.LayerNorm(feature_count)
        self.gru = nn.GRU(feature_count, hidden_size, layers, batch_first=True, dropout=dropout if layers > 1 else 0)
        self.head = nn.Sequential(nn.LayerNorm(hidden_size), nn.Linear(hidden_size, 24), nn.GELU(),
                                  nn.Dropout(dropout), nn.Linear(24, 1))

    def forward(self, sequence):
        output, _ = self.gru(self.input_norm(sequence))
        return self.head(output[:, -1, :]).squeeze(-1)


class BehavioralMLP(nn.Module):
    def __init__(self, feature_count=8, sequence_length=12, dropout=0.20):
        super().__init__()
        size = feature_count * sequence_length
        self.network = nn.Sequential(nn.Flatten(), nn.LayerNorm(size), nn.Linear(size, 128), nn.GELU(), nn.Dropout(dropout),
                                     nn.Linear(128, 48), nn.GELU(), nn.Dropout(dropout), nn.Linear(48, 1))

    def forward(self, sequence):
        return self.network(sequence).squeeze(-1)


MODEL_NAMES = ("lstm", "gru", "mlp")


def build_model(name, feature_count=8, sequence_length=12):
    name = name.lower()
    if name == "lstm":
        return BehavioralLSTM(feature_count)
    if name == "gru":
        return BehavioralGRU(feature_count)
    if name == "mlp":
        return BehavioralMLP(feature_count, sequence_length)
    raise ValueError(f"unknown architecture: {name}; choices={','.join(MODEL_NAMES)}")


# ---------------------------------------------------------------------------------------------
# 3) CHECKPOINT: the file read by the agent
# ---------------------------------------------------------------------------------------------
def save_checkpoint(path, model, architecture, threshold, **extra):
    """Writes a checkpoint that follows the contract. `extra` must only contain plain Python values."""
    torch.save({"model_state": model.state_dict(), "architecture": architecture, "features": list(FEATURE_NAMES),
                "sequence_length": SEQUENCE_LENGTH, "threshold": float(threshold), **extra}, path)


def load_checkpoint(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(_t("ckpt_missing", path=path))
    ckpt = torch.load(path, map_location="cpu", weights_only=True)
    if tuple(ckpt.get("features", FEATURE_NAMES)) != FEATURE_NAMES:
        raise ValueError(_t("ckpt_features"))
    model = build_model(str(ckpt.get("architecture", "lstm")), len(FEATURE_NAMES), int(ckpt.get("sequence_length", SEQUENCE_LENGTH)))
    model.load_state_dict(ckpt["model_state"])
    model.eval()
    return model, ckpt


# ---------------------------------------------------------------------------------------------
# 4) QUICK BENCHMARK: compares LSTM / GRU / MLP on your windows and saves champion.pt
# ---------------------------------------------------------------------------------------------
def _best_threshold(y_true, probs):
    best = (0.5, -1.0)
    for thr in np.arange(0.10, 0.91, 0.02):
        pred = probs >= thr
        tp = int(((y_true == 1) & pred).sum()); fp = int(((y_true == 0) & pred).sum()); fn = int(((y_true == 1) & ~pred).sum())
        f1 = 2 * tp / max(1, 2 * tp + fp + fn)
        if f1 > best[1]:
            best = (float(thr), f1)
    return best


def quick_benchmark(X_train, y_train, X_val, y_val, models=MODEL_NAMES, epochs=25, lr=2e-3, seed=42, out_dir=None):
    """Trains each architecture (class weights, early stopping), picks the threshold on VALIDATION, saves <name>.pt and champion.pt."""
    from torch.utils.data import DataLoader, TensorDataset
    out = Path(out_dir if out_dir is not None else OUT_DIR)
    out.mkdir(parents=True, exist_ok=True)
    xt, yt = torch.tensor(X_train, dtype=torch.float32), torch.tensor(y_train, dtype=torch.float32)
    xv, yv = torch.tensor(X_val, dtype=torch.float32), torch.tensor(y_val, dtype=torch.float32)
    results = {}
    for name in models:
        torch.manual_seed(seed)
        model = build_model(name)
        pos_weight = float((len(yt) - yt.sum()) / max(1.0, float(yt.sum())))
        loss_fn = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pos_weight]))
        optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-3)
        loader = DataLoader(TensorDataset(xt, yt), batch_size=64, shuffle=True)
        best_loss, best_state, bad = 1e9, None, 0
        for epoch in range(epochs):
            model.train()
            for xb, yb in loader:
                optimizer.zero_grad()
                loss_fn(model(xb), yb).backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
            model.eval()
            with torch.inference_mode():
                val_loss = float(loss_fn(model(xv), yv))
            if val_loss < best_loss - 1e-4:
                best_loss, best_state, bad = val_loss, copy.deepcopy(model.state_dict()), 0
            else:
                bad += 1
                if bad >= 5:
                    break
        model.load_state_dict(best_state)
        model.eval()
        with torch.inference_mode():
            probs = torch.sigmoid(model(xv)).numpy()
        thr, f1 = _best_threshold(np.asarray(y_val), probs)
        pred = probs >= thr
        tp = int(((np.asarray(y_val) == 1) & pred).sum()); fp = int(((np.asarray(y_val) == 0) & pred).sum()); fn = int(((np.asarray(y_val) == 1) & ~pred).sum())
        prec, rec = tp / max(1, tp + fp), tp / max(1, tp + fn)
        params = sum(p.numel() for p in model.parameters())
        save_checkpoint(out / f"{name}.pt", model, name, thr, seed=int(seed),
                        validation_metrics={"f1": float(f1), "precision": float(prec), "recall": float(rec)})
        results[name] = {"f1": float(f1), "precision": float(prec), "recall": float(rec), "threshold": thr, "params": params}
        print(_t("bench_row", name=name, f1=f1, prec=prec, rec=rec, thr=thr, params=params))
    champion = max(results, key=lambda k: (results[k]["f1"], -results[k]["params"]))
    model, ckpt = load_checkpoint(out / f"{champion}.pt")
    torch.save(ckpt, out / "champion.pt")
    print(_t("bench_champion", name=champion.upper(), path=out / "champion.pt"))
    return results


# ---------------------------------------------------------------------------------------------
# 5) PLATFORM CALLS
# ---------------------------------------------------------------------------------------------
def _call(path, payload=None, key=False, timeout=10):
    """JSON call. key=True adds your group's API key. Raises RuntimeError with a readable message."""
    headers = {"Content-Type": "application/json"}
    if key:
        if not API_KEY:
            raise RuntimeError(_t("need_key"))
        headers["X-API-Key"] = API_KEY
    body = json.dumps(payload).encode() if payload is not None else None
    for attempt in range(5):
        request = urllib.request.Request(API_URL + path, data=body, headers=headers, method="POST" if payload is not None else "GET")
        try:
            with urllib.request.urlopen(request, timeout=timeout) as response:
                return json.load(response)
        except urllib.error.HTTPError as error:
            if error.code in (429, 502, 503) and attempt < 4:     # too many requests / server restarting: wait, retry
                time.sleep(1.0 + attempt)
                continue
            detail = error.read().decode("utf-8", errors="replace")[:200]
            raise RuntimeError(_t("http", code=error.code, detail=detail)) from None
        except OSError as error:
            if attempt < 2:
                time.sleep(1.0)
                continue
            raise RuntimeError(_t("net", error=error)) from None


def join_class(class_code, name, members):
    """Registers your group in the class (code given by your teacher) and prints your API key ONCE."""
    answer = _call("/api/open/join", {"class_code": class_code, "name": name, "members": members})
    print(_t("joined_class", cls=answer.get("class")))
    print("   ", answer["apiKey"])
    return answer["apiKey"]


def ask_api_key():
    """Asks for the key without writing it anywhere (not in the notebook, not in the outputs)."""
    global API_KEY
    if not API_KEY:
        API_KEY = getpass.getpass(_t("key_help")).strip() or None
    print(_t("key_ok") if API_KEY else _t("key_none"))


# ---------------------------------------------------------------------------------------------
# 6) THE AGENT
# ---------------------------------------------------------------------------------------------
def _probability(model, values):
    with torch.inference_mode():
        return torch.sigmoid(model(torch.tensor([values], dtype=torch.float32))).item()


def run_open_training(checkpoint, player=None, limit=200, batch=20, threshold=None, show=True):
    """OPEN TRAINING (no key, unlimited, nothing graded): your model classifies events and sees TP/TN/FP/FN at once."""
    model, ckpt = load_checkpoint(checkpoint)
    thr = float(threshold if threshold is not None else ckpt.get("threshold", 0.5))
    player = player or GROUP
    print(_t("open_head", player=player))
    seen, stats = 0, {}
    try:
        while seen < limit:
            batch_events = _call(f"/api/open/training/events?count={min(batch, limit - seen)}&player={player}")["events"]
            for event in batch_events:
                steps = ([encode(step) for step in event["history"]] + [encode(event["metrics"])])[-SEQUENCE_LENGTH:]
                values = [[0.0] * len(FEATURE_NAMES)] * (SEQUENCE_LENGTH - len(steps)) + steps
                p = _probability(model, values)
                decision = "block" if p >= thr else "allow"
                result = _call("/api/open/training/decide", {"event_id": event["event_id"], "decision": decision,
                                                              "player": player, "lang": LANG})
                seen += 1
                code = {"VP": "TP", "VN": "TN"}.get(result["outcome"], result["outcome"])
                if show:
                    print(_t("open_line", sw=event["software"], ev=event["event"], dec=decision.upper(), p=p,
                             outcome=_t(code), cat=result["category"]))
            stats = _call(f"/api/open/training/stats?player={player}")
    except KeyboardInterrupt:
        print(_t("stopped"))
    if stats:
        print(_t("summary", n=stats["decisions"], vp=stats["VP"], vn=stats["VN"], fp=stats["FP"], fn=stats["FN"],
                 prec=stats["precision"], rec=stats["recall"], f1=stats["f1"]))
    return stats


def _agent_loop(model, thr, seconds=None, send=True, until_not_mine=False, verbose=False):
    """Core of the live agent: poll telemetry, decide once per event, send the decision."""
    histories = defaultdict(lambda: deque(maxlen=SEQUENCE_LENGTH))
    process = psutil.Process() if psutil else None
    if process:
        process.cpu_percent(None)
    sent = blocked = latency_sum = 0
    started = time.time()
    last_print = 0
    while True:
        cycle = time.perf_counter()
        try:
            snapshot = _call("/api/v1/agent/telemetry", key=True)
            evaluation = snapshot.get("evaluation") or {}
            if until_not_mine and not evaluation.get("mine"):
                return {"sent": sent, "blocked": blocked, "team": snapshot.get("team", {})}
            for software in snapshot.get("telemetry", {}).get("software", []):
                histories[software["id"]].append(encode(software))
            for event in snapshot.get("pendingEvents", []):
                values = padded_sequence(histories[event["softwareId"]], event["metrics"])
                p = _probability(model, values)
                decision = "block" if p >= thr else "allow"
                if not send:
                    print(f"[{event['software']}] {event['event']} -> {decision.upper()} P={p:.3f} (dry-run)")
                    continue
                payload = {"event_id": event["event_id"], "decision": decision,
                           "confidence": p if decision == "block" else 1 - p}
                if process:
                    payload.update({"agent_cpu_percent": min(100.0, process.cpu_percent(None)),
                                    "agent_ram_mb": process.memory_info().rss / 1048576})
                try:
                    answer = _call("/api/v1/agent/decision", payload, key=True)
                except RuntimeError as error:        # 404 expired / 409 already answered / 429: keep going
                    if verbose:
                        print(error)
                    continue
                sent += 1
                blocked += decision == "block"
                latency_sum += answer.get("latencyMs") or 0
                feedback = answer.get("trainingFeedback")
                if feedback and verbose:
                    code = {"VP": "TP", "VN": "TN"}.get(feedback.get("outcome"), feedback.get("outcome"))
                    print(f"[{event['software']}] {event['event']} -> {decision.upper()} | {_t(code)}")
        except RuntimeError as error:
            if verbose:
                print(error)
        if seconds is not None and time.time() - started >= seconds:
            return {"sent": sent, "blocked": blocked, "team": {}}
        if until_not_mine and sent and time.time() - last_print >= 20:
            last_print = time.time()
            left = (_call("/api/v1/agent/evaluation", key=True).get("remainingSeconds") or 0)
            print(_t("progress", n=sent, blocked=blocked, lat=round(latency_sum / max(1, sent)), left=left))
        time.sleep(max(0.0, 0.5 - (time.perf_counter() - cycle)))


def run_live(checkpoint, seconds=30, dry_run=True, threshold=None):
    """(optional) Rehearsal on the server's live session: dry_run=True only predicts, False also answers (TRAINING feedback, no points)."""
    model, ckpt = load_checkpoint(checkpoint)
    thr = float(threshold if threshold is not None else ckpt.get("threshold", 0.5))
    print(_t("live_head", mode="dry-run" if dry_run else "training", seconds=seconds))
    try:
        result = _agent_loop(model, thr, seconds=seconds, send=not dry_run, verbose=True)
        print(result["sent"], "decisions sent")
    except KeyboardInterrupt:
        print(_t("stopped"))


def exam_report():
    """Your TP / TN / FP / FN summary (available once the exam is over)."""
    try:
        report = _call("/api/v1/agent/report", key=True)
    except RuntimeError as error:
        print(_t("report_na", code=str(error)))
        return None
    confusion = report["confusion"]
    print(_t("cm"))
    for key, field in (("cm_tp", "truePositive"), ("cm_tn", "trueNegative"), ("cm_fp", "falsePositive"), ("cm_fn", "falseNegative")):
        print(f"           {_t(key):<28} = {confusion[field]}")
    print(_t("pts", points=report["team"].get("totalPoints"), f1=report["team"].get("f1")))
    return report


def start_exam(checkpoint, threshold=None, confirm=True):
    """Starts YOUR graded exam (once!), runs the agent until the end, then prints your TP / TN / FP / FN."""
    model, ckpt = load_checkpoint(checkpoint)
    thr = float(threshold if threshold is not None else ckpt.get("threshold", 0.5))
    status = _call("/api/v1/agent/evaluation", key=True)
    print(_t("state", state=status["state"], minutes=status["durationMinutes"], used=status["attemptsUsed"], allowed=status["attemptsAllowed"]))
    if status["state"] in ("running", "countdown") and status.get("mine"):
        print(_t("joined"))
    elif status["state"] != "available":
        print(_t("unavailable", state=status["state"], remaining=status.get("remainingSeconds")))
        return None
    else:
        if confirm and input(_t("confirm", minutes=status["durationMinutes"], word=_t("word"))).strip().upper() != _t("word"):
            print(_t("cancel"))
            return None
        started = _call("/api/v1/agent/evaluation/start", {}, key=True)
        print(_t("started", delay=started["delay_seconds"], minutes=started["duration_minutes"]))
    try:
        result = _agent_loop(model, thr, send=True, until_not_mine=True)
    except KeyboardInterrupt:
        print(_t("stopped"))
        return None
    team = result["team"]
    print(_t("done", n=team.get("decisions", 0), points=team.get("totalPoints", 0), f1=team.get("f1", 0),
             fp=team.get("falsePositives", 0), fn=team.get("falseNegatives", 0), lat=team.get("latency", 0)))
    return exam_report()


print("🧰", "toolbox ready" if LANG == "en" else "boîte à outils prête",
      "· encode, padded_sequence, build_model, save_checkpoint, load_checkpoint, quick_benchmark,",
      "run_open_training, run_live, start_exam, exam_report, join_class")

🧰 toolbox ready · encode, padded_sequence, build_model, save_checkpoint, load_checkpoint, quick_benchmark, run_open_training, run_live, start_exam, exam_report, join_class


## 🎟️ Mission 0 — Join the class

The next cell automatically uses the three values entered at the beginning. **One group member runs it once.**

If a key is already loaded, it creates nothing. Otherwise, immediately copy the displayed `tp_live_…` key and clear the output.

**Badge to unlock:** 🔑 team connected.

In [37]:
# 🎟️ Mission 0 — register the team once
if API_KEY:
    print("✅ An API key is already loaded: no new registration.")
elif GROUP == "group_XX" or CLASS_CODE == "CLASS_CODE" or not MEMBERS.strip():
    print("❌ Complete GROUP, MEMBERS and CLASS_CODE in the start-up cell.")
else:
    API_KEY = join_class(CLASS_CODE, GROUP, MEMBERS)
    print("✅ Badge unlocked: team connected. Copy the key, then clear this output.")

✅ An API key is already loaded: no new registration.


## 🧠 Step 1 — Understand the mission (before coding)

The agent receives **behavioural observations**: CPU, RAM, disk reads/writes, network throughput, number of actions per second, process status (`busy`, `running`, `sleeping`) and sometimes connection information (attempts, port, inbound/outbound, known provider). It answers `allow` or `block`.

**What the model has to learn = a behaviour, not a name.** If the AI learns "`CipherBurst.exe` ⇒ malware", it will fail as soon as a piece of malware changes its name.

### Who is "malicious" in the simulator?

The simulator contains **more than 300 fake software programs** generated from **21 behaviour families**. **240 are visible during training** (labelled), **~90 are reserved for the exam**: same families, but other software, other names, other "personalities" (more or less resource-hungry). The **names give nothing away**: `Nimbus Sync` can be a backup tool just as well as a data thief.

#### 🔴 Malicious families

| Family | What sets it apart (check it in the data!) |
|---|---|
| **Ransomware** | Massive disk writes with activity that RAMPS UP (chained encryption). |
| **Cryptocurrency miner** | CPU saturated continuously WITHOUT disk, with a tiny but constant outbound flow to an unknown server. |
| **Data exfiltration** | Reads many files then sends far more than it receives, to unknown IPs. |
| **Botnet beacon** | Almost total silence punctuated by very short, PERIODIC network signals (command and control). |
| **Spyware** | Little CPU but a very high number of actions (keyboard/screen capture) and a small regular outbound flow. |
| **Worm / network scanner** | Huge number of actions and scattered outbound traffic (port scanning) with medium CPU. |
| **Data wiper** | Extreme disk writes and runaway activity, no network: mass file destruction. |
| **Dropper (loader)** | Downloads then writes and launches another program, then long inactivity (looks like an update). |
| **Stealth rootkit** | Almost invisible: slight constant disk activity and a small permanent outbound flow. Hard case. |

#### 🟢 Legitimate families (several of them **look like** the malicious ones)

| Family | What sets it apart (check it in the data!) |
|---|---|
| **Web browser** | Incoming network traffic (pages, downloads) with moderate CPU: normal browser behaviour. |
| **Office suite** | Low, occasional activity (typing, saving): almost no network. |
| **Cloud backup** | Disk reads + network upload, but scheduled and with acknowledgements (non-zero receive traffic): a backup, not a theft. |
| **Build tool** | High CPU and disk in bursts, no network: a compilation (looks like ransomware but stops and does not ramp up). |
| **Video encoder** | Very high, sustained CPU like a miner, BUT with disk reads/writes and lots of memory. |
| **Antivirus (scan)** | Massive disk reads but almost no writes: a scan, not encryption. |
| **Software updater** | Big download then write, short and scheduled: an update (looks like a dropper). |
| **Database** | Regular, balanced activity (stable reads/writes/network): a data server. |
| **System service** | Very low consumption, discreet and regular activity: a system daemon. |
| **Media player / game** | Sustained CPU and memory, incoming network stream (streaming), very few writes. |
| **File synchronisation** | Balanced and regular upload AND download: a two-way sync. |
| **Archiver (zip)** | Short CPU + disk read/write spike (looks like encryption) but it stops quickly and does not accelerate. |

> ⚠️ About **4 % of the observations are misleading**: a piece of malware keeps a low profile for a few seconds, a legitimate program has an unusual spike. No model is perfect — this is intentional.

> 🎓 **If you come from AI**: in cyber, "abnormal" ≠ "malicious". A lot of disk activity is **normal** for a backup. A good detector looks at the **combination** (disk + renames + no known provider) and at the **dynamics over time**.
>
> 🛡️ **If you come from CYBER**: this is **behavioural detection** (like an EDR). The model learns a function that, from a *window* of 12 successive observations, outputs a **probability of being malicious**. That probability is compared with a **threshold** to decide.

### ✍️ Mini-exercise (3 minutes, no code) — `🏷️ CYBER domain`

Before looking at the data, write **your hypothesis**: which 3 signals seem the most discriminating to you? Which kind of legitimate program is likely to be confused with an attack? You will compare with what you see in step 3.

> My hypothesis (edit this cell): …

## 📥 Step 2 — Load the data and plot it

📍 `donnees/open-train-42.jsonl` — **downloaded just below** from the open training sandbox (or provided by the instructor).

The file contains **dozens of sessions** (independent "days" of a simulated workstation), with a `dataset_split` column (`train` / `validation`) that suggests a split **by whole session**.

One line = one event:

```json
{"event_id": "...", "session_id": "...", "stream_id": "...", "timestamp": 1787586009217,
 "event": "network.connect", "phase": "training",
 "metrics": {"cpu": 2.7, "ramMb": 15.3, "diskReadMbps": 0.14, "diskWriteMbps": 0.34,
             "networkRxMbps": 0.0, "networkTxMbps": 0.0, "actionsPerSecond": 6, "status": "running", ...},
 "label": 0, "target_decision": "allow"}
```

| Field | Role | As a feature? |
|---|---|---|
| `event_id` | unique identifier (traceability) | ❌ never |
| `session_id` | one simulation "day" | ❌ used to **split** train/validation |
| `stream_id` | a stream (≈ one anonymised software program) | ❌ used to **group** the sequences |
| `timestamp` | time order | ❌ raw; ✅ for sorting |
| `event` | event type (`network.connect`, `disk.high_io`…) | ⚠️ must be justified |
| `phase` | `training` / `evaluation` | ❌ audit only |
| `metrics.*` | behaviour | ✅ the core of the signal |
| `label` (0/1) and `target_decision` | **the answer** | ❌ **target only** |

> ⚠️ **Classic leakage**: putting `label` or `target_decision` in the features. The model would score 100 % in training and be useless live.

> 🎓 **AI**: `stream_id` looks like a software identifier. Using it as a feature = memorising identity (shortcut learning). In production a new software program would have a new, unknown ID.
> 🛡️ **CYBER**: it is the equivalent of a "block this exact hash" rule: useless against a variant. We want to detect the **behaviour**.

In [7]:
# 🏷️ Mixed — Download the dataset from the OPEN TRAINING SANDBOX (no key needed)
import urllib.request
SESSIONS, TICKS, DATA_SEED = 40, 200, 42        # ✍️ increase them for more data (max 80 sessions)
if not TRAIN_PATH.exists():
    TRAIN_PATH.parent.mkdir(parents=True, exist_ok=True)
    url = f"{API_URL}/api/open/training/dataset?sessions={SESSIONS}&ticks={TICKS}&seed={DATA_SEED}&split=all"
    with urllib.request.urlopen(url, timeout=60) as reponse, open(TRAIN_PATH, "wb") as fichier:
        fichier.write(reponse.read())
    print("✅ downloaded:", TRAIN_PATH)
else:
    print("already present:", TRAIN_PATH, "(delete this file to download another one)")

✅ downloaded: C:\Users\LENOVO\Security_Usescases\donnees\open-train-42.jsonl


In [8]:
# ✍️ AI domain (CYBER students: AI allowed here) — load the dataset WITHOUT modifying it
import pandas as pd

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for bloc in iter(lambda: f.read(1 << 20), b""):
            h.update(bloc)
    return h.hexdigest()

def load_dataset(path):
    '''TODO: return a flat DataFrame (one column per metric), without losing
    event_id / session_id / stream_id / timestamp / event / phase / label / target_decision.'''
    raise NotImplementedError("To complete: load the JSONL and flatten the 'metrics' column")

# raw = load_dataset(TRAIN_PATH)
# DATASET_SHA256 = sha256_file(TRAIN_PATH)
# print(DATASET_SHA256)          # must equal the hash given by the instructor
# raw.head()

<details><summary>💡 Hint (open if you are stuck)</summary>

```python
df = pd.read_json(path, lines=True, convert_dates=False)   # ⚠️ without convert_dates=False, pandas turns "timestamp" into a date and breaks what follows
metrics = pd.json_normalize(df["metrics"])        # dict -> columns (cpu, ramMb, ...)
raw = pd.concat([df.drop(columns=["metrics"]), metrics], axis=1)
raw = raw.sort_values("timestamp").reset_index(drop=True)
```
Keep `raw` as a **raw copy**: then work on `raw.copy()`.
</details>

### ✅ Checkpoint 2

In [9]:
ok = True
ok &= check("`raw` exists", "raw" in globals(), "uncomment the last 4 lines of the previous cell")
if "raw" in globals():
    ok &= check("identity columns kept", {"event_id", "session_id", "stream_id", "timestamp"} <= set(raw.columns))
    ok &= check("metrics flattened (cpu, ramMb…)", {"cpu", "ramMb"} <= set(raw.columns), "use json_normalize")
    ok &= check("label present", "label" in raw.columns)
    ok &= check("hash computed", "DATASET_SHA256" in globals() and len(DATASET_SHA256) == 64)
print("\n➡️ You can move on to step 3." if ok else "\n⏳ Finish step 2 first.")

❌ `raw` exists  → uncomment the last 4 lines of the previous cell

⏳ Finish step 2 first.


## 🔍 Step 3 — Audit the data (like an analyst)

Before any model: **look**. Produce and **comment in writing**:

- number of rows, **sessions**, **streams** (`stream_id`), time range;
- null values per column, real types;
- duplicate `event_id`s;
- legitimate / malicious proportion (`label`);
- min / median / mean / P95 / max of the metrics, **per class**;
- negative or infinite values;
- number of sessions that contain only one class.

**Plots to produce**: distribution of each metric per class, correlations, a time series of 2 streams, disk/actions relationship, outbound network/label.

> ℹ️ **In this file**: a `stream_id` = **one software program within a session**. You will see **several dozen `session_id`s**: this is what makes a real split by session possible (step 5). You can ask for more or different ones by changing `SESSIONS`, `TICKS` or `DATA_SEED` in the download cell.

> 🎓 **AI**: look at the **extreme values** before "cleaning" them. Here the extremes are often **the attacks themselves**: removing them means removing the signal.
> 🛡️ **CYBER**: what data science calls EDA is the **triage** of an incident: first understand what is normal to know what stands out.

In [10]:
# ✍️ AI domain (CYBER students: AI allowed) — audit report
def dataset_report(frame):
    '''TODO: return a serialisable dict with all the measures listed above.'''
    raise NotImplementedError

# report = dataset_report(raw)
# print(json.dumps(report, indent=2, ensure_ascii=False, default=str))

# Minimal assertions to write (uncomment once `raw` exists):
# assert raw["event_id"].is_unique
# assert set(raw["label"].dropna().unique()) <= {0, 1}
# METRICS = ["cpu", "ramMb", "diskReadMbps", "diskWriteMbps", "networkRxMbps", "networkTxMbps", "actionsPerSecond"]
# assert not raw[METRICS].isin([float("inf"), float("-inf")]).any().any()

In [11]:
# ✍️ CYBER domain (AI students: AI allowed) — read the plots like an analyst
# TODO: plot (matplotlib) the distribution of cpu, diskWriteMbps, networkTxMbps, actionsPerSecond
#       split by label, then answer the questions below in the next text cell.
import matplotlib.pyplot as plt
# fig, axes = plt.subplots(2, 2, figsize=(11, 7))
# ...

### ✍️ Your answers (edit this cell) — `🏷️ CYBER domain`

1. Which variables **really** separate the classes? → …
2. Which **legitimate** activity looks like an attack? → …
3. Is a single observation enough, or does the **dynamics** (the history) bring something? → …
4. Was my step 1 hypothesis right? → …
5. Which anomalies did I find and what did I decide (fix / keep / flag / exclude)? → …

### ✅ Checkpoint 3
You can continue if you have: an audit report, at least 4 commented plots and the 5 answers above.

## 🧹 Step 4 — Clean and encode (the "contract" to respect)

The most important point of the whole lab:

> **The preprocessing used during training must be strictly identical to the one used by the live agent.**

That is why the toolbox already provides **the contract** (`encode`, `padded_sequence`). Read its cell (2 minutes):

```python
FEATURE_NAMES = ("cpu", "ram_mb", "disk_read_mbps", "disk_write_mbps",
                 "network_rx_mbps", "network_tx_mbps", "actions_per_second", "busy")
SEQUENCE_LENGTH = 12
encode(metrics)            # dict of metrics -> 8 numbers between 0 and 2
padded_sequence(...)       # history + observation -> window of 12 steps (zeros on the left)
```

### Two tracks

| | **Track A — Lab contract (recommended)** | **Track B — Free** |
|---|---|---|
| Features | the 8 from the toolbox | your own (e.g. deltas, connections…) |
| Training | `quick_benchmark(...)` **or** your code with `build_model` | your code |
| Agent | `run_open_training` / `start_exam` **as they are** | **you write your agent** (starting point: read `_agent_loop` in the toolbox) |
| Risk | low | high: you must reproduce the exact same features on the agent side |

> 💡 If you hesitate: **do Track A first**, up to a clean exam. You can explore Track B afterwards during your free training.

### "Keep / transform / exclude" decision table

| Data | Action | Why |
|---|---|---|
| CPU, RAM, disk, network, actions/s | keep (normalised by `encode`) | behavioural signal |
| `status` (`busy`/`running`/`sleeping`) | encoded as `busy` 0/1 | context |
| missing values | neutral value (0) + note how many | do not throw away a whole session |
| extreme values | **clip** (already done: capped at 2× the scale) | do not erase the attacks |
| `event_id`, `session_id`, `stream_id` | exclude from the tensor, keep for grouping | leakage / memorisation |
| `label`, `target_decision` | **target only** | direct leakage |
| `timestamp` | sort, then forget | avoid learning the date |

> 🛡️ **CYBER**: the scale (`/100`, `/512`, `/50`, `/250`) puts every metric on roughly 0–1 so the network is not "blinded" by a large unit (RAM in MB would crush CPU in %). The cap at 2 prevents an extreme spike from breaking the computation while still staying visible as "very abnormal".
> 🎓 **AI**: here the normalisation is **fixed** (no `StandardScaler.fit`), so there is no statistics leakage. If you switch to Track B with a fitted scaler, fit it **on the train set only** and save it together with the model.

In [12]:
# ✍️ AI domain (CYBER students: AI allowed) — encode your data with THE project contract
# FEATURE_NAMES, SEQUENCE_LENGTH and encode come from the toolbox (lab contract)
print(FEATURE_NAMES, SEQUENCE_LENGTH)

# TODO: create a DataFrame `clean` sorted by (session_id, stream_id, timestamp), with
#   - the 8 feature columns computed with encode(<dict of the row's metrics>)
#   - label (0/1), session_id, stream_id kept for grouping / splitting
#   - a cleaning log: how many rows were touched/excluded and why
# Hint: encode() expects a dict with the keys cpu, ramMb, diskReadMbps, ... status

('cpu', 'ram_mb', 'disk_read_mbps', 'disk_write_mbps', 'network_rx_mbps', 'network_tx_mbps', 'actions_per_second', 'busy') 12


## ✂️ Step 5 — Split train / validation WITHOUT leakage

Golden rule: **split by session (or by stream), never row by row.**

Why? Events of the same session are **correlated**: if row 100 is in the train set and row 101 in the validation set, the model has "already seen" almost the same thing. You would get a great score… and a misleading one.

Mandatory order:
1. the **official final test** is already kept by the instructor (it is your live exam);
2. split your file into **train / validation** by `session_id` (≈ 80 / 20);
3. any parameter learning (scaler, class weights…) is done on the **train set only**.

> ℹ️ The open dataset contains `dataset_split` (whole sessions already distributed). You can use it **or** redo your own split by `session_id` with `GroupShuffleSplit` (recommended: you control the seed). In every case: **no window may cross two groups**, and you write it in your report.
>
> ⛔ If one day you work on a **single-session** file, do **not** split by `stream_id` (one stream can be 95% malicious and another 3%: your two sides would not have the same proportion). Use the temporal split per stream with a gap of 12 events, shown as a comment in the cell below.

> 🎓 **AI**: this is a `GroupKFold`/`GroupShuffleSplit`, not a random `train_test_split`.
> 🛡️ **CYBER**: imagine testing an antivirus on variants of a malware *already in its signatures*: it looks perfect. You must test it on families **it has never seen**.

In [13]:
# ✍️ AI domain (CYBER students: AI allowed) — grouped and reproducible split
# from sklearn.model_selection import GroupShuffleSplit
# groups = clean["session_id"]            # whole sessions: never mixed between train and validation
# splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
# train_idx, val_idx = next(splitter.split(clean, groups=groups))
# train_groups, val_groups = set(groups.iloc[train_idx]), set(groups.iloc[val_idx])
# assert train_groups.isdisjoint(val_groups), "LEAKAGE: a group is on both sides!"
# print(len(train_idx), len(val_idx), clean.iloc[train_idx]["label"].mean(), clean.iloc[val_idx]["label"].mean())

# --- Example file (a single session): TEMPORAL split per stream, with a gap of 12 events
# import numpy as np
# train_idx, val_idx = [], []
# for _, part in clean.groupby(["session_id", "stream_id"]):          # clean is already sorted by timestamp
#     pos = part.index.to_numpy(); cut = int(len(pos) * 0.8)
#     train_idx += list(pos[:cut]); val_idx += list(pos[cut + SEQUENCE_LENGTH:])   # gap = no shared window
# train_idx, val_idx = np.array(train_idx), np.array(val_idx)
# print(len(train_idx), len(val_idx), clean.loc[train_idx, "label"].mean(), clean.loc[val_idx, "label"].mean())

### ✅ Checkpoint 5
You have: two sets whose groups are **disjoint**, **both classes present in each**, and the positive rate of each side printed. If a class is missing from validation, change the seed *in a documented way* — never move an isolated row.

## 🪟 Step 6 — Build the sequences (windows of 12)

The model does not look at an isolated observation but at **the last 12 of the same stream** → tensor `(N, 12, 8)`.

Rule (identical to the live agent, see `padded_sequence` in the toolbox):
- group by `(session_id, stream_id)`, sort by `timestamp`;
- for each event: take the 11 previous ones **from the same stream** + the current event;
- if the history is short: complete with **zeros on the left** (padding);
- **window label = label of its last event**;
- **never** mix two streams or two sessions in a window.

> 🛡️ **CYBER**: this is **event correlation**: a burst of renames *after* a file enumeration is far more suspicious than either alone. Order matters.
> 🎓 **AI**: left padding with zeros is what the agent does when a stream starts. If your training only has full windows, your agent will meet cases live that it has never seen (start of a stream).

In [14]:
# ✍️ AI domain (CYBER students: AI allowed)
import numpy as np

def make_sequences(frame, sequence_length=SEQUENCE_LENGTH):
    '''TODO: return X (N, 12, 8) float32, y (N,) float32 and groups (N,) for the split.
    Group by (session_id, stream_id), sort by timestamp, sliding window, left padding.'''
    raise NotImplementedError

# X_train, y_train, g_train = make_sequences(clean.iloc[train_idx])
# X_val,   y_val,   g_val   = make_sequences(clean.iloc[val_idx])
# assert X_train.ndim == 3 and X_train.shape[1:] == (12, 8)
# assert np.isfinite(X_train).all() and len(X_train) == len(y_train)

<details><summary>💡 Hint</summary>

```python
for (_, _), g in frame.groupby(["session_id", "stream_id"], sort=False):
    g = g.sort_values("timestamp")
    vecs = g[list(FEATURE_NAMES)].to_numpy(dtype="float32")
    for i in range(len(g)):
        window = vecs[max(0, i - 11): i + 1]
        pad = np.zeros((12 - len(window), 8), dtype="float32")
        X.append(np.vstack([pad, window])); y.append(g["label"].iloc[i])
```
</details>

## 📏 Step 7 — Baselines: beat simple things first

An LSTM that does not beat a 3-line rule is useless. Compute **on the validation set**:

| Baseline | What it measures |
|---|---|
| `allow-all` | the cost of false negatives (no attack blocked) |
| `block-all` | the cost of false positives (everything is blocked) |
| **business rule** (e.g. ≥ 2 signals among cpu>45, disk>6, network>4, actions>100) | what a SIEM rule would achieve |
| **logistic regression** on the last observation | a simple statistical model |

For each one: **confusion matrix, precision, recall, F1 and business score** (see the formula in Step 9).

> 🎓 **AI**: do not use accuracy alone. If 40% of the rows are malicious, `allow-all` has 60% accuracy while being useless.
> 🛡️ **CYBER**: the business rule is your **expertise** turned into a baseline. If you know attacks well, this is where you can beat a naive model — and then **understand why the AI does better or worse**.

In [15]:
# ✍️ Mixed — baselines
# from sklearn.linear_model import LogisticRegression
# from sklearn.metrics import confusion_matrix, precision_recall_fscore_support
#
# def rule_baseline(X):               # X: (N, 12, 8) -> 0/1 decisions on the LAST step
#     last = X[:, -1, :]               # values are normalised: cpu=0.45 <=> 45 %
#     signals = (last[:, 0] > .45).astype(int) + (last[:, 3] > .12) + (last[:, 5] > .08) + (last[:, 6] > .40)
#     return (signals >= 2).astype(int)
#
# results = {}   # name -> dict(precision, recall, f1, FP, FN, business_score)

## 🤖 Step 8 — Train models

Recommended minimum for your report: **1 baseline + 1 non-recurrent model (MLP) + 2 temporal models (LSTM, GRU)**.

### 📍 Quick path — `quick_benchmark`

The toolbox compares **LSTM / GRU / MLP** on **your** windows (weighted classes, early stopping, threshold chosen on the **validation set**) and saves `models/group_XX/lstm.pt`, `gru.pt`, `mlp.pt` and **`champion.pt`**. Run the cell below once your sequences are built (Step 6):

In [16]:
# 🏷️ Mixed — compare LSTM / GRU / MLP in one go (needs X_train, y_train, X_val, y_val from Step 6)
if "X_train" in globals() and GROUP != "group_XX":
    benchmark = quick_benchmark(X_train, y_train, X_val, y_val, epochs=25)
else:
    print("⏳ Fill in your GROUP (startup cell) and finish Step 6 to get X_train / X_val.")

⏳ Fill in your GROUP (startup cell) and finish Step 6 to get X_train / X_val.


> ℹ️ The score shown is measured on **your validation set** (split **by session**): it is **not** your official live exam.
> ℹ️ More data? Re-run the download with `SESSIONS = 80` (Step 2). No synthetic data to add: you have **real** labelled sessions.

### 📍 Free path — your own PyTorch loop

Reuse `build_model("gru", 8, 12)` from the toolbox or write your own architecture (then see "Track B" above).

In [17]:
# ✍️ AI domain (CYBER students: AI allowed) — experiment sheet, then training loop
# One change at a time! Record each trial (hypothesis, config, result, conclusion).
EXPERIMENT = {
    "hypothesis": "TO_COMPLETE",
    "architecture": "gru",          # lstm | gru | mlp (see the toolbox)
    "seed": SEED, "sequence_length": 12,
    "batch_size": 64, "learning_rate": 2e-3, "weight_decay": 1e-3,
    "max_epochs": 30, "patience": 5, "grad_clip": 1.0,
}

# import torch; from torch import nn; from torch.utils.data import DataLoader, TensorDataset
# torch.manual_seed(EXPERIMENT["seed"])
# model = build_model(EXPERIMENT["architecture"], 8, 12)
# pos_weight = (len(y_train) - y_train.sum()) / max(1, y_train.sum())        # minority class, TRAIN only
# loss_fn = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([float(pos_weight)]))
# optimizer = torch.optim.AdamW(model.parameters(), lr=EXPERIMENT["learning_rate"], weight_decay=EXPERIMENT["weight_decay"])
# for epoch in range(EXPERIMENT["max_epochs"]):
#     model.train()      # TODO: loop over the DataLoader, zero_grad, loss, backward, clip_grad_norm_, step
#     model.eval()       # TODO: measure the VALIDATION loss with torch.inference_mode()
#     # TODO: keep the best state (validation), early stopping after `patience` epochs without progress

<details><summary>💡 Training hints & pitfalls</summary>

- The model output is a **logit**: apply `torch.sigmoid` to get a probability between 0 and 1.
- `BCEWithLogitsLoss(pos_weight=…)` expects **logits**, not probabilities.
- `model.train()` / `model.eval()`: dropout does not behave the same way. Forgetting `eval()` = unstable results.
- **Pick the best epoch on the validation set**, never on anything else.
- A `nan` loss ⇒ an infinite value in the data, a learning rate that is too high, or an unclipped gradient.
- If validation is excellent but the live exam is poor: look for **leakage** (identity, label, row-by-row split).
</details>

> 🛡️ **CYBER**: *epoch* = one complete pass over the training data. *Early stopping* = we stop when performance on **unseen** data (validation) stops improving — otherwise the AI learns by heart.
> 🎓 **AI**: remember that the live simulator is not your dataset: the more your train/val curves diverge, the more you risk poor generalisation.

### Comparison table to fill in (keep **all** trials, failures included)

| Model | Parameters | Threshold | Val F1 | Val recall | Val precision | FP | FN | Business score | P95 latency | Decision |
|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|---|
| allow-all | — | — | | | | | | | | |
| block-all | — | — | | | | | | | | |
| business rule | — | — | | | | | | | | |
| logistic regression | | | | | | | | | | |
| MLP | | | | | | | | | | |
| LSTM | | | | | | | | | | |
| GRU | | | | | | | | | | |

## 🎚️ Step 9 — Choose the threshold with the **real cost**

The model outputs `P(malicious)`. The agent blocks if `P ≥ threshold`. The 0.5 threshold is **not** automatically a good one.

With the exam scoring, the **business score** of a threshold on the validation set is:

```text
score = 10·TP  +  6·TN  −  8·FP  −  12·FN
(TP = threats blocked (API code `VP`), TN = legitimate allowed (API code `VN`), FP = legitimate blocked, FN = threats let through)
```

→ Try thresholds from 0.10 to 0.90, plot score / F1 / recall / FP / FN as a function of the threshold, pick the best one **on the validation set**, then **freeze it**.

> 🎓 **AI**: this is threshold optimisation under asymmetric costs, not F1 maximisation. The two can give different thresholds: justify your choice.
> 🛡️ **CYBER**: lowering the threshold = alerting more easily (more attacks caught, more false alarms). It is the sensitivity setting of a sensor.
> ⚠️ The threshold is written in the checkpoint (`threshold`) and read by the agent. **Once you have started the exam, it no longer changes.**

In [18]:
# ✍️ Mixed — threshold → cost curve
# probs_val = ...        # P(malicious) of the model on X_val (sigmoid of the logits)
def business_score(y_true, y_pred):
    vp = int(((y_true == 1) & (y_pred == 1)).sum()); vn = int(((y_true == 0) & (y_pred == 0)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum()); fn = int(((y_true == 1) & (y_pred == 0)).sum())
    return 10 * vp + 6 * vn - 8 * fp - 12 * fn, dict(VP=vp, VN=vn, FP=fp, FN=fn)

# thresholds = np.arange(0.10, 0.91, 0.02)
# TODO: for each threshold, compute business_score, F1, recall; plot; choose THRESHOLD
# THRESHOLD = ...

## 🧪 Step 10 — Robustness and speed

Before the live exam, check that your model does not collapse. To measure (on the validation set):

| Test | How | Expected |
|---|---|---|
| Sensor noise ±5 % | multiply the metrics by `1 ± 0.05` | F1 drops by **< 0.10** |
| Missing values | set 10 % of the inputs to 0 | no crash, reasonable degradation |
| Legitimate spike (backup) | build/find a backup sequence | **allowed** if possible |
| Start of a stream | almost empty sequence (padding) | probability between 0 and 1, no crash |
| Inference latency | 1,000 predictions of 1 sequence, after warm-up | median and **P95** (indicative target < 150 ms) |
| Resources | size of the `.pt`, number of parameters, RAM | recorded in the table |

> 🛡️ **CYBER**: this is the equivalent of an **evasion test**: an attacker (or plain noise) can shift the values a little. A detector that collapses with 5 % noise is too fragile.
> 🎓 **AI**: latency counts for the **speed** points (≤150 ms = 5 pts) and the agent must be able to handle several events per second without falling behind.

In [19]:
# ✍️ AI domain (CYBER students: AI allowed) — P95 latency
# import torch, time
# x = torch.tensor(X_val[:1], dtype=torch.float32)
# temps = []
# with torch.inference_mode():
#     for _ in range(50): model(x)                 # warm-up
#     for _ in range(1000):
#         t = time.perf_counter(); model(x); temps.append((time.perf_counter() - t) * 1000)
# print("median ms:", np.median(temps), "| P95 ms:", np.percentile(temps, 95))

## 📦 Step 11 — Export your champion (the agent's contract)

The provided agent (`run_open_training`, `start_exam`) loads a `.pt` file that **must contain** these keys (`save_checkpoint` writes them for you):

| Key | Content | Required |
|---|---|:-:|
| `model_state` | `model.state_dict()` | ✅ |
| `architecture` | `"lstm"`, `"gru"` or `"mlp"` | ✅ |
| `features` | exactly `FEATURE_NAMES` (same order) | ✅ |
| `sequence_length` | `12` | ✅ |
| `threshold` | your frozen threshold | ✅ (otherwise 0.5) |
| `metrics`, `validation_metrics`, `seed`, `epochs`… | traceability | recommended |

If you used `quick_benchmark`, **`models/group_XX/champion.pt` already follows this contract**: you only need to (optionally) replace `threshold` with the one you chose in step 9 (cell below).

> ⚠️ Custom architecture (other than lstm/gru/mlp) or other features ⇒ the provided agent cannot load it: you need your own agent (Track B).

In [20]:
# ✍️ Mixed — write / update the checkpoint with YOUR threshold
import torch

CHECKPOINT = OUT_DIR / "champion.pt"      # or your own file

def set_threshold(path, threshold):
    '''Updates the threshold in a checkpoint produced by quick_benchmark.'''
    ckpt = torch.load(path, map_location="cpu", weights_only=True)
    ckpt["threshold"] = float(threshold)
    torch.save(ckpt, path)
    print("threshold", threshold, "written to", path)

# set_threshold(CHECKPOINT, THRESHOLD)

# --- If you export from your own loop:
# torch.save({"model_state": model.state_dict(), "architecture": EXPERIMENT["architecture"],
#             "features": FEATURE_NAMES, "sequence_length": SEQUENCE_LENGTH,
#             "threshold": float(THRESHOLD), "seed": SEED, "validation_metrics": {...}}, CHECKPOINT)
# ⚠️ Put plain PYTHON float() values in validation_metrics (not numpy.float64) and no sklearn objects:
#    otherwise torch.load(..., weights_only=True) refuses the file (UnpicklingError) and the agent will not be able to load it.

In [21]:
# 🏷️ Mixed — COLD CHECK of the checkpoint (run it: it changes nothing)
# Reproduces exactly what the agent (load_checkpoint) does when loading.
def verify_checkpoint(path):
    FN, SL = FEATURE_NAMES, SEQUENCE_LENGTH
    path = Path(path)
    if not check(f"checkpoint present ({path.name})", path.exists(), f"not found: {path}"):
        return False
    ckpt = torch.load(path, map_location="cpu", weights_only=True)
    ok = check("key model_state", "model_state" in ckpt)
    ok &= check("known architecture (lstm/gru/mlp)", ckpt.get("architecture") in ("lstm", "gru", "mlp"),
                f"architecture={ckpt.get('architecture')!r} → you need your own agent")
    ok &= check("features == FEATURE_NAMES (same order)", tuple(ckpt.get("features", FN)) == FN)
    ok &= check("sequence_length == 12", int(ckpt.get("sequence_length", SL)) == SL)
    thr = ckpt.get("threshold")
    ok &= check(f"threshold defined and within ]0,1[ (= {thr})", isinstance(thr, float) and 0 < thr < 1)
    if ok:
        model = build_model(ckpt["architecture"], len(FN), SL)
        model.load_state_dict(ckpt["model_state"]); model.eval()
        with torch.inference_mode():
            p = torch.sigmoid(model(torch.zeros(1, SL, len(FN)))).item()
        ok &= check(f"prediction bounded in [0,1] (p={p:.3f})", 0.0 <= p <= 1.0)
    return bool(ok)

# verify_checkpoint(CHECKPOINT)

## 🏋️ Step 12 — Train **as much as you want** before the exam

This is the heart of the design: **no limit on attempts until you start the exam.**

### The improvement loop

```text
        ┌──────────────────────────────────────────────────────────────┐
        ▼                                                              │
 1. OPEN TRAINING SANDBOX  →  2. ANALYSE THE ERRORS (FP? FN?)  →  3. IMPROVE / RETRAIN
 (TP/TN/FP/FN feedback         (which family fools you?)             (ONE change at a time)
  on every decision)
```

### 1) The open training sandbox: train with the right answer, **no key needed**

Your model classifies events from a simulated workstation, **then the verdict comes back immediately**:

```text
Nimbus Sync · network.upload → ALLOW (P=0.01) | FN ❌ threat missed — family: Data exfiltration
Onyx Core · file.write → ALLOW (P=0.00) | TN ✅ legitimate allowed — family: Office suite
[SUMMARY] 100 decisions | TP=36 TN=60 FP=0 FN=4 | precision=1.00 recall=0.90 F1=0.95
```

* **TP / TN** (API codes `VP` / `VN`) = good decisions · **FP / FN** = mistakes.
* The **family** tells you which type of software fooled you → that is your improvement lead (e.g. "all the botnet beacons get through" → look at periodicity).
* No limit, no ranking, no points. `limit=500` stops after 500 decisions.

> 🎓 **AI**: this is **interactive supervised learning**: you measure your model on data it has *never seen* and could not memorise. But if you fix your model for *these* specific events, you overfit to the open training sandbox. Look for **families**, not individual cases.
> 🛡️ **CYBER**: this is a *purple team* exercise: for each attack family, you see whether your detection misses it (FN) or cries wolf (FP).

In [22]:
# 🏷️ Mixed — OPEN TRAINING SANDBOX: no key, unlimited, nothing is graded (re-run it as often as you like)
if CHECKPOINT.exists():
    stats = run_open_training(CHECKPOINT, player=GROUP, limit=200)
else:
    print("⏳ Export your champion first (step 11) or run quick_benchmark (step 8).")

⏳ Export your champion first (step 11) or run quick_benchmark (step 8).


The routes of the open training sandbox (useful if you write your own agent), all **without a key**:

| Route | Role |
|---|---|
| `GET /api/open/catalog` | the 240 training software items **with their true nature** |
| `GET /api/open/training/events?count=20&player=NAME` | events to classify (+ `history`: the previous measurements of the same software) |
| `POST /api/open/training/decide` | `{event_id, decision, player}` → `outcome` VP/VN/FP/FN (API codes for TP/TN/FP/FN), `why` |
| `GET /api/open/training/stats?player=NAME` | your TP/TN/FP/FN counters, precision, recall, F1 |
| `GET /api/open/training/dataset?...` | multi-session dataset (already used in step 2) |

To test another threshold **without retraining**: `run_open_training(CHECKPOINT, threshold=0.6)`.

### 2) (optional) Dress rehearsal on the server's "live" session

The server runs a simulated workstation continuously. With **your key** (step 13), `run_live` either predicts without sending anything (`dry_run=True`, to test latency and that everything works) or answers with `TRAINING` feedback, no points:

In [23]:
# 🏷️ Mixed — (optional, needs your key: step 13) dress rehearsal on the server's live session
# run_live(CHECKPOINT, seconds=30, dry_run=True)

### 3) Analyse your errors

Classify each error: **false positive** (a legitimate program blocked) or **false negative** (a threat let through)? Behaviour absent from the training set? Badly built window? Threshold? Bug?

> ⚠️ **Do not turn the feedback into a rule by software name** ("block X"). The live exam uses other data: you would memorise and fail.

### 4) Retrain and start again
One change at a time, record it in your table. You can repeat this cycle **as many times as you want**.

## 🚦 Step 13 — Are you ready? (the "GO / NO-GO" checklist)

Start the exam **only if everything is ticked**. Reminder: **a single attempt**.

- [ ] My best model is in `models/group_XX/champion.pt` and `verify_checkpoint` is green ✅
- [ ] My threshold was chosen **on the validation set** and written into the checkpoint
- [ ] My agent ran in **dry-run** without errors and in **training** at least once
- [ ] My inference latency is low (P95 < 150 ms, indicative)
- [ ] My key is **not** in the notebook nor in Git
- [ ] My **API key** is entered (🔑 cell) and I know how to run the exam cell of step 14
- [ ] No other agent uses my key (otherwise `409` errors)
- [ ] My PC stays plugged in, sleep disabled, stable network for **5 minutes + 10 s**
- [ ] My team knows that **nobody touches anything** during the exam
- [ ] I re-read the "How you will be assessed" section

### 🔑 Your API key (masked input)

Run the next cell: a **masked** field asks for your group's key (`tp_live_…`). It stays in memory for the duration of the session and is **written nowhere** (neither in the notebook, nor in its outputs).

In [ ]:
# 🏷️ Mixed — enter your API key (masked field)
ask_api_key()

In [ ]:
# 🏷️ Mixed — PRE-FLIGHT: run this cell just before deciding (it does NOT start the exam)
def evaluation_status():
    code, data = api("/api/v1/agent/evaluation")
    return code, data

print("=== PRE-FLIGHT ===")
go = True
go &= check("API key loaded", bool(API_KEY and API_KEY.startswith("tp_live_")), "run the \"🔑 enter your API key\" cell above")
go &= check("platform reachable", api("/api/health")[0] == 200, "check TP_URL / network")
try:
    go &= verify_checkpoint(CHECKPOINT)
except NameError:
    go &= check("checkpoint verified", False, "run the cells of step 11")

code, status = evaluation_status()
if code == 200:
    etat = status["state"]
    print(f"\nState of your exam: {etat} | duration {status['durationMinutes']} min | "
          f"attempts {status['attemptsUsed']}/{status['attemptsAllowed']}")
    messages = {
        "available": "✅ You can start it whenever you want.",
        "closed": "⛔ The instructor closed your class's exam: keep training until it reopens.",
        "exhausted": "⛔ You already used your attempt. Ask the instructor whether another one is possible.",
        "disabled": "⛔ Self-service exam is disabled: the instructor will start it.",
        "countdown": "⏱️ Your exam is starting: start/keep your agent running!",
        "running": "🔴 Your exam is RUNNING. Do not touch anything.",
    }
    print(messages.get(etat, etat))
    go &= etat in ("available", "countdown", "running")
else:
    go &= check("exam status readable", False, f"HTTP {code} {status}")

print("\n" + ("🟢 GO — if your checklist is ticked, you can start the exam (step 14)."
              if go else "🔴 NO-GO — fix the ❌ above."))

## 🔴 Step 14 — Start YOUR 5-minute exam

> ⚠️ **Last warning**: the exam is **graded**, plays out **in real time**, and **cannot be replayed** (one attempt by default). From the click on: **no more retraining, no threshold change, no code change.**

### Recommended method (a single cell, with confirmation)

▶️ When your checklist is ticked, **uncomment the last line** of the cell below and run it. It stays "running" for ~5 minutes: this is normal, **do not interrupt the kernel**.

In [ ]:
# 🔴 GRADED EXAM — one attempt by default. Uncomment ONLY when you are ready.
# start_exam(CHECKPOINT)

What the function does:
1. reads your state (`GET /api/v1/agent/evaluation`) and refuses if the exam is not available;
2. asks you to **type `EXAM`** to confirm (a safeguard against accidental clicks; `confirm=False` skips it);
3. calls **`POST /api/v1/agent/evaluation/start`** → countdown of ~10 s;
4. runs your agent for the 5 minutes;
5. stops **by itself** at the end and prints your summary (decisions, points, F1, FP, FN, latency) **and your TP / TN / FP / FN matrix**.

### "By hand" method (if you wrote your own agent, or to understand the API)

The "path" to hit is:

| | |
|---|---|
| **Method / route** | `POST {TP_URL}/api/v1/agent/evaluation/start` |
| **Header** | `X-API-Key: <your key>` |
| **Body (optional)** | `{"delay_seconds": 10}` (3 to 120) |
| **Status** | `GET {TP_URL}/api/v1/agent/evaluation` |

```bash
# See the state
curl -H "X-API-Key: $TP_API_KEY" "$TP_URL/api/v1/agent/evaluation"

# Start (⚠️ irreversible)
curl -X POST -H "X-API-Key: $TP_API_KEY" -H "Content-Type: application/json" \
     -d '{"delay_seconds": 10}' "$TP_URL/api/v1/agent/evaluation/start"
```
```powershell
$h = @{ "X-API-Key" = $env:TP_API_KEY }
Invoke-RestMethod "$env:TP_URL/api/v1/agent/evaluation" -Headers $h
Invoke-RestMethod "$env:TP_URL/api/v1/agent/evaluation/start" -Method Post -Headers $h `
  -ContentType "application/json" -Body '{"delay_seconds": 10}'
```

Then your agent has to loop: `GET /api/v1/agent/telemetry` (every `pollAfterMs`) → for each item of `pendingEvents` → **one** answer `POST /api/v1/agent/decision`.

### Possible answers when starting

| Code | `detail` | Meaning | What to do |
|---:|---|---|---|
| `202` | — | Exam accepted, countdown started | Let the agent run |
| `401` | `invalid_api_key` | Invalid key | Check the key / ask for a new one |
| `409` | `exam_closed` | The instructor closed your class's exam | Keep training, wait for it to open |
| `409` | `evaluation_already_running` | Yours is already running | Do not start again: start/keep your agent |
| `409` | `evaluation_attempts_exhausted` | Attempt already used | See the instructor |
| `409` | `self_evaluation_disabled` | Self-service exam disabled | The instructor will start it |
| `429` | `rate_limit_exceeded` | Too many requests | Slow down (`pollAfterMs`) |

### During the 5 minutes: what should you do?

**Nothing.** Watch the cell output, breathe. If the agent prints occasional network errors, it keeps going by itself (it handles temporary errors). If the agent really crashes, **re-run exactly the same cell**: it detects that your exam is already running, **joins it without restarting it** and does not use a new attempt.

> 🛡️ **CYBER**: this is an automated incident-response exercise: no human in the loop, decisions within seconds, service availability to preserve.
> 🎓 **AI**: this is **inference in production**: unseen data (*distribution shift*), latency constraint, no way to fix things afterwards.

In [ ]:
# 🏷️ Mixed — (optional) follow your exam from the notebook, read-only
# You can re-run this cell during the exam: it changes nothing. You have NOTHING else to do.
code, st = api("/api/v1/agent/evaluation")
print(code, st)
if code == 200 and st.get("mine"):
    print(f"Your exam: {st['state']} — about {st.get('remainingSeconds')} s left")

## 📊 Step 15 — After the exam: analyse, explain, hand in

The exam ends automatically. Your agent prints your **TP / TN / FP / FN matrix** (API codes `VP` / `VN` / `FP` / `FN`); you can read it again at any time with:

```bash
curl -H "X-API-Key: $TP_API_KEY" "$TP_URL/api/v1/agent/report"
```

Once your attempt is used up, the report also contains **the detail of every decision** (software, event, what you decided, what was expected, TP/TN/FP/FN). The instructor sees the same detail per group and per class. **You write the analysis.**

### To do in the minutes that follow
1. Write down the **summary** printed by the agent (decisions, points, F1, FP, FN, latency).
2. Compare with your **validation**: gap ≤ 0.10? If not, why (generalisation, leakage, surprises)?
3. Without rewriting history: list **at least three mistakes** (FP and FN) and explain them.
4. Suggest improvements **and the protocol to test them** (you can no longer test them on the exam).

### To hand in
- this completed notebook (plots, answers, comparison table);
- `ai_usage_log.md` (generative-AI usage log);
- `models/group_XX/` (checkpoints + `benchmark.json`) — without the key or the test;
- a short **report** (dataset provenance with its hash, audit, split, baselines, models, threshold, robustness, resources, live results, mistakes, limits).

## ✍️ Personal conclusion (to write)

Explain: the model you kept and why · the main source of **false positives** · the main source of **false negatives** · the effect of **temporal context** · the **threshold / recall** trade-off · the **runtime** cost · what would fail on a real workstation · the extra data you would need.

*A good conclusion separates what was **measured**, what is **assumed** and what remains **unknown**.*

> My conclusion (edit this cell): …

## 🆘 Quick troubleshooting

| Symptom | Likely cause | Fix |
|---|---|---|
| `OSError: WinError 1114 … c10.dll` (Windows) | `torch` imported after pandas / sklearn | restart the kernel and run the start-up cell **first** (it imports `torch` first) |
| `ModuleNotFoundError` (torch, pandas…) | dependencies not installed | run the "📦 Installing the dependencies" cell, then **restart the kernel** |
| "API key loaded" ❌ at step 13 | the 🔑 cell was not run | run `ask_api_key()` and paste your key |
| `HTTP 0` / platform unreachable | wrong URL, firewall, server down | check `TP_URL`, try `/api/health` in the browser |
| Very slow (≈ 2 s per call) on Windows | `localhost` resolved over IPv6 then falling back to IPv4 | on a local server, use `http://127.0.0.1:3000` in `TP_URL` |
| `HTTP 401` | invalid / renewed key | ask for a new key |
| `HTTP 409` on a decision | you already answered that event, or 2 agents use your key | stop the second agent |
| `HTTP 404` on a decision | event expired (> ~15 s) | speed up inference |
| `HTTP 429` | too many requests | respect `pollAfterMs` |
| `409 exam_closed` when starting | the class exam is closed | wait for the instructor to open it |
| `409 evaluation_attempts_exhausted` | attempt already used | see the instructor |
| Nothing to decide (key mode) | no exam running | use `run_open_training(...)` (no key) or start your exam |
| `unknown architecture` or `incompatible features` | checkpoint outside the contract | review step 11 |
| Perfect validation, poor live results | data **leakage** | review steps 4–5 (identity, label, split) |
| Only one class in validation | unlucky split | change the seed (documented) or use more sessions |
| `loss = nan` | infinite value / learning rate too high | check the data, lower the learning rate, keep the clipping |

### Going further
📍 Interactive API documentation: `https://ml-cyber.magro.pro/docs`.